# Durability — 4. NN training (global fck, rh, cov, t $\to$ lambda)

Trains one `MLPRegressor` for `lambda 1` and one for `lambda 2` on the dataset stacked by
[`03_generate_dataset_nn.ipynb`](03_generate_dataset_nn.ipynb). Unlike the PCE stage
(`02_train_pce.ipynb`, one model per time step), this fits a single global model that also takes
$t$ as an input — query it with any $(f_{ck}, RH, c, t)$ combination, no need to pick a PCE for a
specific time step first.

`lambda 3` / `lambda 4` are **not** modelled here — read them back from the emulator dataset
directly, same convention as the benchmark.

Functions come from [`functions.py`](../functions.py): `train_and_validate_nn_lambda_durability`.
Prediction plots and the KL-divergence check are in
[`04_train_nn_plot.ipynb`](04_train_nn_plot.ipynb).

## 1. Libraries

In [5]:
%matplotlib inline
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))

import dill
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib as mpl
mpl.rcParams.update({
                        'font.family': 'serif',
                        'mathtext.fontset': 'cm',
                        'axes.unicode_minus': False
                    })
from sklearn.model_selection import train_test_split

from functions import *

## 2. Config

`n_latent_samples`, `installation_year`, `co2_scenario`, `cement_type` and `exposure_conditions`
must match [`03_generate_dataset_nn.ipynb`](03_generate_dataset_nn.ipynb) — together they name the
file being loaded.

In [6]:
n_latent_samples    = 2500      # must match stage 3 — it names the file being loaded
installation_year   = 1980
co2_scenario        = "SSP2-4.5"
cement_type         = 3
exposure_conditions = 2

feature_cols = ['fck', 'rh', 'cov', 'Time (years)']
target_cols  = ['lambda 1', 'lambda 2']

test_frac          = 0.2
hidden_layer_sizes = (64, 64)
max_iter           = 500
n_iter_no_change   = 15
random_state       = 42

fig_size   = (5, 4)      # size of each individual figure, in inches
fig_format = 'png'       # format each figure is saved in ('pdf', 'png', ...)
fig_dpi    = 300         # resolution the figure is saved at (dots per inch)

label_fontsize = 14   # font size of the axis labels
tick_fontsize  = 12   # font size of the tick numbers

xlim = None   # e.g. (-5, 5) to fix the axis; None = auto-scaled to the data, per lambda
ylim = None   # e.g. (-5, 5) to fix the axis; None = auto-scaled to the data, per lambda

## 3. Load the stacked dataset

In [7]:
tag = f'install_{installation_year}_cement_{cement_type}_exposure_{exposure_conditions}_co2_{co2_scenario}'
df_nn = load_pkl(f'{n_latent_samples}_dataset_nn_durability_{tag}.pkl')

print(f"Loaded {len(df_nn)} rows")
df_nn.head()

Loaded 25000 rows


,fck,rh,cov,Time (years),lambda 1,lambda 2,lambda 3,lambda 4
0,39.596275,67.675794,28.860733,0.0,28.482822,0.333963,0.201702,0.136161
1,25.572027,61.299428,24.170547,0.0,23.855135,0.406760,0.201702,0.136161
2,41.419597,76.104319,34.633872,0.0,34.166028,0.274092,0.201702,0.136161
3,48.780600,44.718265,34.519324,0.0,34.090914,0.276286,0.201702,0.136161
4,45.880345,20.524925,23.495412,0.0,23.160538,0.417394,0.201702,0.136161


## 4. Train and validate

In [8]:
print("="*60)
print("TRAINING THE DURABILITY NN")
print("="*60)

result = train_and_validate_nn_lambda_durability(
                                                    df_nn=df_nn,
                                                    feature_cols=feature_cols,
                                                    target_cols=target_cols,
                                                    test_frac=test_frac,
                                                    hidden_layer_sizes=hidden_layer_sizes,
                                                    max_iter=max_iter,
                                                    n_iter_no_change=n_iter_no_change,
                                                    random_state=random_state,
                                                    n_latent_samples=n_latent_samples,
                                                    installation_year=installation_year,
                                                    cement_type=cement_type,
                                                    exposure_conditions=exposure_conditions,
                                                    co2_scenario=co2_scenario,
                                                    output_dir='.',
                                                 )

result['statistics']

TRAINING THE DURABILITY NN

----------------------------------------
TRAINING NN LAMBDA MODELS
----------------------------------------
  20000 train rows, 5000 val rows
  lambda 1: R² = 0.999564, MSE = 0.10946, iterations = 45
  lambda 2: R² = 0.999214, MSE = 0.00001, iterations = 75
The NN models, scaler and validation stats have been saved!


,MSE lambda 1,R² lambda 1,MSE lambda 2,R² lambda 2
0,0.109458,0.999564,0.000007,0.999214
